In [1]:

import gradio as gr
import numpy as np
import cv2
from PIL import Image
import io
import base64
from sklearn.cluster import KMeans
import svgwrite

def ai_vectorize(image):
    # Resize image for performance
    max_dim = 512
    height, width = image.shape[:2]
    if max(height, width) > max_dim:
        scale = max_dim / float(max(height, width))
        image = cv2.resize(image, (int(width * scale), int(height * scale)))

    # Convert to RGB if not already
    if image.shape[2] == 4:
        image = cv2.cvtColor(image, cv2.COLOR_BGRA2BGR)

    img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    flat_img = img_rgb.reshape((-1, 3))
    
    # KMeans for palette control
    k = 8  # Number of colors
    kmeans = KMeans(n_clusters=k, random_state=42).fit(flat_img)
    clustered = kmeans.cluster_centers_[kmeans.labels_]
    clustered_img = clustered.reshape(img_rgb.shape).astype(np.uint8)

    # Edge detection for shape fitting
    gray = cv2.cvtColor(clustered_img, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    contours, _ = cv2.findContours(edges, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    # Generate SVG
    height, width = gray.shape
    dwg = svgwrite.Drawing(size=(width, height))
    for cnt in contours:
        if len(cnt) < 10:
            continue
        path = "M " + " L ".join(f"{pt[0][0]},{pt[0][1]}" for pt in cnt)
        path += " Z"
        color = tuple(clustered_img[cnt[0][0][1], cnt[0][0][0]])
        hex_color = f"rgb({color[0]},{color[1]},{color[2]})"
        dwg.add(dwg.path(d=path, fill=hex_color, stroke="none"))

    svg_bytes = dwg.tostring().encode()
    svg_output = svg_bytes.decode()

    # PNG preview
    preview_img = Image.fromarray(clustered_img)
    preview_bytes = io.BytesIO()
    preview_img.save(preview_bytes, format='PNG')
    preview_bytes.seek(0)

    return svg_output, preview_bytes, svg_bytes, "data:image/svg+xml;base64," + base64.b64encode(svg_bytes).decode()

with gr.Blocks() as demo:
    gr.Markdown("## 🎨 AI Vectorizer (Multi-colour Smooth Vector Output)")

    with gr.Row():
        input_img = gr.Image(type="numpy", label="Upload Bitmap (PNG/JPG)")

    with gr.Row():
        svg_code = gr.Textbox(label="SVG Output Code")
        preview_png = gr.Image(label="Vector Preview (PNG)")
    
    with gr.Row():
        download_pdf = gr.File(label="Download PDF")
        download_svg = gr.File(label="Download SVG")

    output_html = gr.HTML()
    
    generate_btn = gr.Button("Generate Vector")

    def wrap_vectorizer(img):
        if img is None:
            return "Error: No image provided", None, None, ""
        
        svg_str, png_io, svg_io, svg_url = ai_vectorize(img)
        
        # Save files
        with open("vector_output.svg", "w", encoding="utf-8") as f:
            f.write(svg_str)
        Image.open(png_io).save("vector_preview.png")
        
        return svg_str, "vector_preview.png", "vector_output.svg", f"<iframe src='{svg_url}' width='100%' height='500px'></iframe>"

    generate_btn.click(
        fn=wrap_vectorizer,
        inputs=[input_img],
        outputs=[svg_code, preview_png, download_svg, output_html]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
